# Week 17 Day 06 Learning Notebook

This day notebook is fully executable and aligned to the lesson content.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 17 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 17 Day 05: Factor robustness
- Previous lesson file: content/week-17/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 17 Day 07: Portfolio Project
- Next lesson file: content/week-17/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Rebuild IC/decay calculations from scratch
- Review neutralization settings
- Document crowding warning indicators

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


## Runnable Day Example
Run this example for Week 17 Day 06, inspect outputs, then complete the quiz.

In [2]:
# Revision sprint demo: rebuild weekly core diagnostics
np.random.seed(1706)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()

summary = pd.DataFrame({
    'annual_return': returns.mean() * 252,
    'annual_vol': returns.std() * np.sqrt(252),
    'max_drawdown': (prices / prices.cummax() - 1).min(),
})
summary['sharpe_proxy'] = (summary['annual_return'] - 0.03) / summary['annual_vol'].replace(0, np.nan)
summary = summary.sort_values('sharpe_proxy', ascending=False)

print('Revision diagnostics (best risk-adjusted first):')
summary.round(4)


Revision diagnostics (best risk-adjusted first):


,annual_return,annual_vol,max_drawdown,sharpe_proxy
Ticker,,,,
AAPL,0.277000,0.306000,-0.385200,0.807300
QQQ,0.205600,0.238800,-0.351200,0.735400
SPY,0.151700,0.193100,-0.337200,0.630400


## Week 17 Day 06 Quiz

Topic: **Revision Sprint**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [3]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 123.000
price_t = 124.353
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Revision Sprint')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.011000)
print('  gross_return_expected  =', 1.011000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 123.0
  P_t    : 124.353
  r_t    : 0.011 => 1.10%
  1+r_t  : 1.011

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Revision Sprint
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.011
  gross_return_expected  = 1.011
